<a href="https://colab.research.google.com/github/VinayaSharada/FinancialAnalyticsCourse/blob/main/PortfolioOpt/portfolio_optimization_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Portfolio Optimization Demo

This notebook demonstrates Modern Portfolio Theory and portfolio optimization techniques using synthetic financial data.

**Course**: Financial Analytics Masters Course  
**Topic**: Portfolio Optimization  
**Author**: AI Assistant  
**Date**: July 2025  

## 📚 Setup and Installation

In [ ]:
# Install required packages
!pip install pandas numpy matplotlib seaborn scipy -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ All packages installed and imported successfully!")

## 📥 Load Synthetic Data

We'll load the synthetic data generated from the previous notebook.

In [ ]:
# Load data from GitHub repository or generate if not available
import os

DATA_URL = "https://raw.githubusercontent.com/VinayaSharada/FinancialAnalyticsCourse/main/PortfolioOpt/syntheticdata.csv"
LOCAL_FILE = "syntheticdata.csv"

# Try to load from local file first, then from URL
try:
    if os.path.exists(LOCAL_FILE):
        print(f"📁 Loading data from local file: {LOCAL_FILE}")
        df = pd.read_csv(LOCAL_FILE)
    else:
        print(f"🌐 Loading data from GitHub repository...")
        df = pd.read_csv(DATA_URL)
        
    df['date'] = pd.to_datetime(df['date'])
    print(f"✅ Data loaded successfully!")
    print(f"📊 Dataset shape: {df.shape}")
    print(f"📅 Date range: {df['date'].min()} to {df['date'].max()}")
    print(f"🏢 Number of assets: {df['symbol'].nunique()}")
    
except Exception as e:
    print(f"❌ Error loading data: {e}")
    print("\n💡 Please ensure you have run the data generation notebook first!")
    
# Display first few rows
display(df.head())

## 🏗️ Portfolio Optimizer Class

In [ ]:
class PortfolioOptimizer:
    """
    Portfolio optimization using Modern Portfolio Theory.
    
    This class implements various portfolio optimization techniques including:
    - Mean-variance optimization
    - Efficient frontier calculation
    - Sharpe ratio maximization
    - Risk parity
    """
    
    def __init__(self, data):
        """
        Initialize the portfolio optimizer.
        
        Parameters:
        -----------
        data : pd.DataFrame
            Financial data with columns: date, symbol, daily_return, etc.
        """
        self.data = data
        self.prepare_data()
        
    def prepare_data(self):
        """Prepare data for portfolio optimization."""
        # Pivot to get returns matrix
        self.returns_data = self.data.pivot(index='date', columns='symbol', values='daily_return')
        self.returns_data = self.returns_data.dropna()
        
        # Calculate annualized statistics
        self.mean_returns = self.returns_data.mean() * 252  # Annualized returns
        self.cov_matrix = self.returns_data.cov() * 252     # Annualized covariance
        self.risk_free_rate = self.data['risk_free_rate'].iloc[0] * 252  # Annualized
        self.assets = list(self.returns_data.columns)
        
        print(f"✅ Data prepared for {len(self.assets)} assets")
        print(f"📊 Return statistics:")
        print(f"   • Mean return range: {self.mean_returns.min():.2%} to {self.mean_returns.max():.2%}")
        print(f"   • Average volatility: {np.sqrt(np.diag(self.cov_matrix)).mean():.2%}")
        print(f"   • Risk-free rate: {self.risk_free_rate:.2%}")
    
    def calculate_portfolio_performance(self, weights):
        """
        Calculate portfolio return, volatility, and Sharpe ratio.
        
        Parameters:
        -----------
        weights : array-like
            Portfolio weights
            
        Returns:
        --------
        tuple
            (return, volatility, sharpe_ratio)
        """
        weights = np.array(weights)
        
        # Portfolio return
        portfolio_return = np.sum(weights * self.mean_returns)
        
        # Portfolio volatility
        portfolio_variance = np.dot(weights.T, np.dot(self.cov_matrix, weights))
        portfolio_volatility = np.sqrt(portfolio_variance)
        
        # Sharpe ratio
        sharpe_ratio = (portfolio_return - self.risk_free_rate) / portfolio_volatility
        
        return portfolio_return, portfolio_volatility, sharpe_ratio
    
    def optimize_sharpe_ratio(self):
        """
        Optimize portfolio for maximum Sharpe ratio.
        
        Returns:
        --------
        dict
            Optimization results
        """
        n_assets = len(self.assets)
        
        # Objective function (negative Sharpe ratio for minimization)
        def negative_sharpe(weights):
            _, volatility, sharpe = self.calculate_portfolio_performance(weights)
            return -sharpe
        
        # Constraints
        constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})  # Weights sum to 1
        bounds = tuple((0, 1) for _ in range(n_assets))  # Long-only
        
        # Initial guess (equal weights)
        initial_guess = np.array([1/n_assets] * n_assets)
        
        # Optimize
        result = minimize(negative_sharpe, initial_guess, method='SLSQP',
                        bounds=bounds, constraints=constraints)
        
        # Calculate performance
        optimal_weights = result.x
        ret, vol, sharpe = self.calculate_portfolio_performance(optimal_weights)
        
        return {
            'weights': optimal_weights,
            'return': ret,
            'volatility': vol,
            'sharpe_ratio': sharpe,
            'success': result.success
        }
    
    def optimize_minimum_variance(self):
        """
        Optimize portfolio for minimum variance.
        
        Returns:
        --------
        dict
            Optimization results
        """
        n_assets = len(self.assets)
        
        # Objective function (portfolio variance)
        def portfolio_variance(weights):
            return np.dot(weights.T, np.dot(self.cov_matrix, weights))
        
        # Constraints
        constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
        bounds = tuple((0, 1) for _ in range(n_assets))
        
        # Initial guess
        initial_guess = np.array([1/n_assets] * n_assets)
        
        # Optimize
        result = minimize(portfolio_variance, initial_guess, method='SLSQP',
                        bounds=bounds, constraints=constraints)
        
        # Calculate performance
        optimal_weights = result.x
        ret, vol, sharpe = self.calculate_portfolio_performance(optimal_weights)
        
        return {
            'weights': optimal_weights,
            'return': ret,
            'volatility': vol,
            'sharpe_ratio': sharpe,
            'success': result.success
        }
    
    def calculate_efficient_frontier(self, n_portfolios=100):
        """
        Calculate the efficient frontier.
        
        Parameters:
        -----------
        n_portfolios : int, default 100
            Number of portfolios to calculate
            
        Returns:
        --------
        pd.DataFrame
            Efficient frontier data
        """
        # Target returns
        min_ret = self.mean_returns.min()
        max_ret = self.mean_returns.max()
        target_returns = np.linspace(min_ret, max_ret, n_portfolios)
        
        efficient_portfolios = []
        n_assets = len(self.assets)
        
        for target_ret in target_returns:
            # Objective function (minimize variance)
            def portfolio_variance(weights):
                return np.dot(weights.T, np.dot(self.cov_matrix, weights))
            
            # Constraints
            constraints = [
                {'type': 'eq', 'fun': lambda x: np.sum(x) - 1},  # Weights sum to 1
                {'type': 'eq', 'fun': lambda x: np.sum(x * self.mean_returns) - target_ret}  # Target return
            ]
            bounds = tuple((0, 1) for _ in range(n_assets))
            
            # Initial guess
            initial_guess = np.array([1/n_assets] * n_assets)
            
            try:
                result = minimize(portfolio_variance, initial_guess, method='SLSQP',
                                bounds=bounds, constraints=constraints)
                
                if result.success:
                    weights = result.x
                    ret, vol, sharpe = self.calculate_portfolio_performance(weights)
                    
                    efficient_portfolios.append({
                        'return': ret,
                        'volatility': vol,
                        'sharpe_ratio': sharpe,
                        'weights': weights
                    })
            except:
                continue
        
        return pd.DataFrame(efficient_portfolios)

print("✅ PortfolioOptimizer class defined successfully!")

## 🚀 Initialize Portfolio Optimizer

In [ ]:
# Initialize the portfolio optimizer
optimizer = PortfolioOptimizer(df)

print("\n📊 Portfolio Universe Summary:")
print(f"• Number of assets: {len(optimizer.assets)}")
print(f"• Date range: {optimizer.returns_data.index.min()} to {optimizer.returns_data.index.max()}")
print(f"• Trading days: {len(optimizer.returns_data)}")

## 🎯 Portfolio Optimization

In [ ]:
print("🚀 Running portfolio optimizations...")

# Optimize for maximum Sharpe ratio
print("\n📈 Optimizing for Maximum Sharpe Ratio...")
sharpe_optimal = optimizer.optimize_sharpe_ratio()
print(f"✅ Optimal Sharpe ratio: {sharpe_optimal['sharpe_ratio']:.4f}")

# Optimize for minimum variance
print("\n📉 Optimizing for Minimum Variance...")
min_var_optimal = optimizer.optimize_minimum_variance()
print(f"✅ Minimum volatility: {min_var_optimal['volatility']:.4f}")

# Calculate efficient frontier
print("\n🔄 Calculating Efficient Frontier...")
efficient_frontier = optimizer.calculate_efficient_frontier(n_portfolios=50)
print(f"✅ Calculated {len(efficient_frontier)} efficient portfolios")

print("\n🎉 All optimizations completed successfully!")

## 📊 Results Summary

In [ ]:
print("="*80)
print("PORTFOLIO OPTIMIZATION RESULTS")
print("="*80)

print(f"\n🎯 OPTIMAL PORTFOLIOS SUMMARY")
print("-" * 40)

print(f"\n📈 Maximum Sharpe Ratio Portfolio:")
print(f"   • Expected Return: {sharpe_optimal['return']:.2%}")
print(f"   • Volatility: {sharpe_optimal['volatility']:.2%}")
print(f"   • Sharpe Ratio: {sharpe_optimal['sharpe_ratio']:.4f}")

print(f"\n📉 Minimum Variance Portfolio:")
print(f"   • Expected Return: {min_var_optimal['return']:.2%}")
print(f"   • Volatility: {min_var_optimal['volatility']:.2%}")
print(f"   • Sharpe Ratio: {min_var_optimal['sharpe_ratio']:.4f}")

# Top holdings
print(f"\n🏆 TOP HOLDINGS - Maximum Sharpe Ratio Portfolio:")
sharpe_weights = pd.Series(sharpe_optimal['weights'], index=optimizer.assets)
top_holdings = sharpe_weights.nlargest(5)
for asset, weight in top_holdings.items():
    if weight > 0.01:  # Only show weights > 1%
        print(f"   • {asset}: {weight:.1%}")

print(f"\n🛡️ TOP HOLDINGS - Minimum Variance Portfolio:")
minvar_weights = pd.Series(min_var_optimal['weights'], index=optimizer.assets)
top_holdings = minvar_weights.nlargest(5)
for asset, weight in top_holdings.items():
    if weight > 0.01:
        print(f"   • {asset}: {weight:.1%}")

# Efficient frontier stats
print(f"\n📊 EFFICIENT FRONTIER STATISTICS:")
print(f"   • Number of efficient portfolios: {len(efficient_frontier)}")
print(f"   • Return range: {efficient_frontier['return'].min():.2%} to {efficient_frontier['return'].max():.2%}")
print(f"   • Volatility range: {efficient_frontier['volatility'].min():.2%} to {efficient_frontier['volatility'].max():.2%}")
print(f"   • Maximum Sharpe ratio: {efficient_frontier['sharpe_ratio'].max():.4f}")

## 📈 Efficient Frontier Visualization

In [ ]:
# Plot efficient frontier
plt.figure(figsize=(12, 8))

# Plot efficient frontier
plt.scatter(efficient_frontier['volatility'], efficient_frontier['return'], 
           c=efficient_frontier['sharpe_ratio'], cmap='viridis', alpha=0.7, s=50)
plt.colorbar(label='Sharpe Ratio')

# Plot optimal portfolios
plt.scatter(sharpe_optimal['volatility'], sharpe_optimal['return'], 
           color='red', s=200, marker='*', label='Max Sharpe Ratio', edgecolors='black')
plt.scatter(min_var_optimal['volatility'], min_var_optimal['return'], 
           color='blue', s=200, marker='*', label='Min Variance', edgecolors='black')

# Plot individual assets
individual_returns = optimizer.mean_returns.values
individual_volatilities = np.sqrt(np.diag(optimizer.cov_matrix))
plt.scatter(individual_volatilities, individual_returns, 
           color='gray', alpha=0.6, s=30, label='Individual Assets')

plt.xlabel('Volatility (Standard Deviation)', fontsize=12)
plt.ylabel('Expected Return', fontsize=12)
plt.title('Efficient Frontier - Portfolio Optimization', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 🥧 Portfolio Composition Charts

In [ ]:
def plot_portfolio_composition(portfolio_result, title, optimizer):
    """Plot portfolio composition pie chart."""
    weights = portfolio_result['weights']
    
    # Only show assets with weight > 1%
    significant_weights = weights[weights > 0.01]
    significant_assets = [optimizer.assets[i] for i, w in enumerate(weights) if w > 0.01]
    
    if len(significant_weights) > 10:
        # Group smaller weights
        sorted_indices = np.argsort(significant_weights)[::-1]
        top_weights = significant_weights[sorted_indices[:9]]
        top_assets = [significant_assets[i] for i in sorted_indices[:9]]
        other_weight = significant_weights[sorted_indices[9:]].sum()
        
        plot_weights = list(top_weights) + [other_weight]
        plot_assets = top_assets + ['Others']
    else:
        plot_weights = significant_weights
        plot_assets = significant_assets
    
    plt.figure(figsize=(10, 8))
    colors = plt.cm.Set3(np.linspace(0, 1, len(plot_weights)))
    
    wedges, texts, autotexts = plt.pie(plot_weights, labels=plot_assets, autopct='%1.1f%%',
                                      colors=colors, startangle=90)
    
    plt.title(f'{title}\nExpected Return: {portfolio_result["return"]:.2%}, '
             f'Volatility: {portfolio_result["volatility"]:.2%}, '
             f'Sharpe Ratio: {portfolio_result["sharpe_ratio"]:.3f}', fontsize=12)
    
    plt.axis('equal')
    plt.tight_layout()
    plt.show()

# Plot portfolio compositions
plot_portfolio_composition(sharpe_optimal, "Maximum Sharpe Ratio Portfolio", optimizer)
plot_portfolio_composition(min_var_optimal, "Minimum Variance Portfolio", optimizer)

## 📊 Risk-Return Analysis

In [ ]:
# Risk-return scatter plot for individual assets
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Individual asset risk-return
ax1.scatter(individual_volatilities, individual_returns, alpha=0.7, s=60)
for i, asset in enumerate(optimizer.assets):
    ax1.annotate(asset, (individual_volatilities[i], individual_returns[i]), 
                xytext=(5, 5), textcoords='offset points', fontsize=8)

ax1.set_xlabel('Volatility')
ax1.set_ylabel('Expected Return')
ax1.set_title('Individual Asset Risk-Return Profile')
ax1.grid(True, alpha=0.3)

# Sharpe ratio comparison
sharpe_ratios = (optimizer.mean_returns - optimizer.risk_free_rate) / np.sqrt(np.diag(optimizer.cov_matrix))
ax2.bar(range(len(sharpe_ratios)), sharpe_ratios.values, alpha=0.7)
ax2.set_xlabel('Assets')
ax2.set_ylabel('Sharpe Ratio')
ax2.set_title('Individual Asset Sharpe Ratios')
ax2.set_xticks(range(len(sharpe_ratios)))
ax2.set_xticklabels(sharpe_ratios.index, rotation=45, ha='right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print top and bottom performers
print("\n🏆 TOP PERFORMING ASSETS (by Sharpe Ratio):")
top_assets = sharpe_ratios.nlargest(5)
for asset, ratio in top_assets.items():
    print(f"   • {asset}: {ratio:.4f}")

print("\n⚠️ BOTTOM PERFORMING ASSETS (by Sharpe Ratio):")
bottom_assets = sharpe_ratios.nsmallest(5)
for asset, ratio in bottom_assets.items():
    print(f"   • {asset}: {ratio:.4f}")

## 🔍 Portfolio Comparison Analysis

In [ ]:
# Compare different portfolio strategies
equal_weights = np.ones(len(optimizer.assets)) / len(optimizer.assets)
equal_ret, equal_vol, equal_sharpe = optimizer.calculate_portfolio_performance(equal_weights)

# Market cap weighted portfolio (assuming larger market cap = better)
market_caps = df.groupby('symbol')['market_cap'].first()
market_caps = market_caps.reindex(optimizer.assets)
market_weights = market_caps / market_caps.sum()
market_ret, market_vol, market_sharpe = optimizer.calculate_portfolio_performance(market_weights)

# Create comparison DataFrame
comparison_data = {
    'Portfolio Strategy': ['Equal Weight', 'Market Cap Weight', 'Max Sharpe Ratio', 'Min Variance'],
    'Expected Return': [equal_ret, market_ret, sharpe_optimal['return'], min_var_optimal['return']],
    'Volatility': [equal_vol, market_vol, sharpe_optimal['volatility'], min_var_optimal['volatility']],
    'Sharpe Ratio': [equal_sharpe, market_sharpe, sharpe_optimal['sharpe_ratio'], min_var_optimal['sharpe_ratio']]
}

comparison_df = pd.DataFrame(comparison_data)

print("\n📊 PORTFOLIO STRATEGY COMPARISON")
print("="*70)
display(comparison_df.round(4))

# Visualization of comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Return comparison
axes[0].bar(comparison_df['Portfolio Strategy'], comparison_df['Expected Return'], alpha=0.7)
axes[0].set_title('Expected Returns Comparison')
axes[0].set_ylabel('Expected Return')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, alpha=0.3)

# Volatility comparison
axes[1].bar(comparison_df['Portfolio Strategy'], comparison_df['Volatility'], alpha=0.7, color='orange')
axes[1].set_title('Volatility Comparison')
axes[1].set_ylabel('Volatility')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, alpha=0.3)

# Sharpe ratio comparison
axes[2].bar(comparison_df['Portfolio Strategy'], comparison_df['Sharpe Ratio'], alpha=0.7, color='green')
axes[2].set_title('Sharpe Ratio Comparison')
axes[2].set_ylabel('Sharpe Ratio')
axes[2].tick_params(axis='x', rotation=45)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 🎓 Student Exploration Guide

### 🔧 Parameter Tuning Exercises:

1. **Risk Tolerance**: 
   - Modify constraints to include maximum position limits (e.g., no asset > 20%)
   - Add sector diversification constraints

2. **Investment Horizons**: 
   - Change the risk-free rate to see impact on optimal portfolios
   - Adjust the return calculation period (daily vs monthly returns)

3. **Optimization Methods**: 
   - Try different optimization algorithms (e.g., 'trust-constr')
   - Implement risk parity optimization

4. **Advanced Constraints**: 
   - Add turnover constraints for transaction costs
   - Implement ESG (Environmental, Social, Governance) scoring

### 📊 Analysis Exercises:

1. **Sensitivity Analysis**:
   - How do optimal portfolios change with different risk-free rates?
   - What happens if you exclude the best/worst performing assets?

2. **Rolling Window Analysis**:
   - Optimize portfolios using different time periods
   - Analyze portfolio stability over time

3. **Sector Analysis**:
   - Group assets by sector and analyze sector allocation
   - Create sector-neutral portfolios

### 🏦 Banking Applications:

This portfolio optimization framework can be used by banks for:

1. **💰 ASSET MANAGEMENT**:
   - Optimize client portfolios based on risk tolerance
   - Create model portfolios for different investor profiles
   - Rebalance portfolios to maintain target allocations

2. **🏛️ TREASURY MANAGEMENT**:
   - Optimize bank's own investment portfolio
   - Manage liquidity across different asset classes
   - Hedge interest rate and credit risks

3. **📋 PRODUCT DEVELOPMENT**:
   - Design structured products and investment funds
   - Create risk-targeted investment solutions
   - Develop robo-advisor algorithms

4. **🎯 RISK MANAGEMENT**:
   - Set position limits and concentration limits
   - Stress test portfolios under different scenarios
   - Calculate Value-at-Risk (VaR) for trading books

5. **💼 WEALTH MANAGEMENT**:
   - Provide personalized investment advice
   - Create tax-efficient portfolio strategies
   - Implement ESG (Environmental, Social, Governance) constraints

## 🔬 Advanced Analytics (Optional)

In [ ]:
# Calculate additional risk metrics
def calculate_var(returns, confidence_level=0.05):
    """Calculate Value at Risk"""
    return np.percentile(returns, confidence_level * 100)

def calculate_cvar(returns, confidence_level=0.05):
    """Calculate Conditional Value at Risk (Expected Shortfall)"""
    var = calculate_var(returns, confidence_level)
    return returns[returns <= var].mean()

def calculate_maximum_drawdown(cumulative_returns):
    """Calculate Maximum Drawdown"""
    peak = np.maximum.accumulate(cumulative_returns)
    drawdown = (cumulative_returns - peak) / peak
    return drawdown.min()

print("\n🔬 ADVANCED RISK ANALYTICS")
print("="*50)

# Calculate portfolio returns for risk metrics
sharpe_portfolio_returns = optimizer.returns_data.dot(sharpe_optimal['weights'])
minvar_portfolio_returns = optimizer.returns_data.dot(min_var_optimal['weights'])
equal_portfolio_returns = optimizer.returns_data.dot(equal_weights)

# Calculate cumulative returns
sharpe_cumret = (1 + sharpe_portfolio_returns).cumprod()
minvar_cumret = (1 + minvar_portfolio_returns).cumprod()
equal_cumret = (1 + equal_portfolio_returns).cumprod()

# Risk metrics
portfolios = {
    'Max Sharpe': sharpe_portfolio_returns,
    'Min Variance': minvar_portfolio_returns,
    'Equal Weight': equal_portfolio_returns
}

cumulative_returns = {
    'Max Sharpe': sharpe_cumret,
    'Min Variance': minvar_cumret,
    'Equal Weight': equal_cumret
}

risk_metrics = []
for name, returns in portfolios.items():
    var_5 = calculate_var(returns, 0.05)
    cvar_5 = calculate_cvar(returns, 0.05)
    max_dd = calculate_maximum_drawdown(cumulative_returns[name])
    
    risk_metrics.append({
        'Portfolio': name,
        'VaR (5%)': var_5,
        'CVaR (5%)': cvar_5,
        'Max Drawdown': max_dd,
        'Skewness': returns.skew(),
        'Kurtosis': returns.kurtosis()
    })

risk_df = pd.DataFrame(risk_metrics)
display(risk_df.round(4))

# Plot cumulative returns
plt.figure(figsize=(12, 6))
for name, cumret in cumulative_returns.items():
    plt.plot(cumret.index, cumret.values, label=name, linewidth=2)

plt.title('Cumulative Returns Comparison', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Cumulative Return')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 💾 Export Results

In [ ]:
# Export optimization results
results_summary = {
    'max_sharpe_weights': dict(zip(optimizer.assets, sharpe_optimal['weights'])),
    'min_var_weights': dict(zip(optimizer.assets, min_var_optimal['weights'])),
    'max_sharpe_performance': {k: v for k, v in sharpe_optimal.items() if k != 'weights'},
    'min_var_performance': {k: v for k, v in min_var_optimal.items() if k != 'weights'},
    'efficient_frontier': efficient_frontier.to_dict('records')
}

# Save to CSV
portfolio_weights = pd.DataFrame({
    'Asset': optimizer.assets,
    'Max_Sharpe_Weight': sharpe_optimal['weights'],
    'Min_Variance_Weight': min_var_optimal['weights'],
    'Equal_Weight': equal_weights,
    'Market_Cap_Weight': market_weights.values
})

portfolio_weights.to_csv('portfolio_weights.csv', index=False)
efficient_frontier.to_csv('efficient_frontier.csv', index=False)
comparison_df.to_csv('portfolio_comparison.csv', index=False)

print("✅ Results exported to CSV files:")
print("   • portfolio_weights.csv")
print("   • efficient_frontier.csv")
print("   • portfolio_comparison.csv")

# For Google Colab users - download the files
try:
    from google.colab import files
    print("\n📥 Downloading result files...")
    files.download('portfolio_weights.csv')
    files.download('efficient_frontier.csv')
    files.download('portfolio_comparison.csv')
    print("✅ Downloads initiated!")
except ImportError:
    print("\n📁 Files saved locally in current directory")
    print("💡 If running locally, you can find the files in your current directory")

## 📚 Key Takeaways

### 🎯 **Modern Portfolio Theory Principles**:
1. **Diversification**: Combining assets reduces portfolio risk without proportionally reducing return
2. **Efficient Frontier**: The set of optimal portfolios offering the highest expected return for each level of risk
3. **Risk-Return Trade-off**: Higher returns generally require accepting higher risk
4. **Sharpe Ratio**: Measures risk-adjusted returns, helping identify the best risk-return combination

### 💡 **Practical Applications**:
1. **Asset Allocation**: Determine optimal mix of assets based on investor preferences
2. **Risk Management**: Quantify and control portfolio risk exposure
3. **Performance Evaluation**: Benchmark portfolio performance against optimal allocations
4. **Investment Strategy**: Guide tactical and strategic asset allocation decisions

### ⚠️ **Limitations to Consider**:
1. **Historical Data**: Optimization relies on past performance which may not predict future results
2. **Static Analysis**: Assumes constant correlations and volatilities over time
3. **Transaction Costs**: Real-world implementation involves costs not captured in the model
4. **Liquidity Constraints**: Some assets may not be easily tradeable in required quantities

### 🚀 **Next Steps for Advanced Learning**:
1. **Dynamic Portfolio Optimization**: Implement time-varying optimization
2. **Alternative Risk Measures**: Explore downside risk, VaR-based optimization
3. **Multi-Period Optimization**: Consider multiple investment horizons
4. **Machine Learning Integration**: Use ML for return prediction and portfolio construction

---

**📧 Questions or Issues?**  
This notebook is part of the Financial Analytics Masters Course.  
For support, please refer to the course materials or contact your instructor.